# Model Experiments: Basic CNN vs U-Net Image Colorization

This notebook covers:
1. Model architectures: Basic CNN (Baseline) vs U-Net (Primary Model).
2. Training configuration with L1 loss and Adam optimizer.
3. Running inference on test images.
4. Side-by-side visual and quantitative evaluation (MAE, PSNR, SSIM).
5. Comparative findings.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from models.basic_cnn import BasicCNN
from models.unet import UNet
from data.celebA_dataset import get_dataloaders
from utils.checkpoint import load_checkpoint
from utils.metrics import calculate_all_metrics
from utils.visualization import plot_comparison_grid
from preprocessing.color_space import lab_to_rgb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")

## 1. Inspect Model Architectures & Parameter Counts

In [ ]:
basic_model = BasicCNN(in_channels=1, out_channels=2).to(device)
unet_model = UNet(in_channels=1, out_channels=2).to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Basic CNN Trainable Parameters: {count_params(basic_model):,}")
print(f"U-Net     Trainable Parameters: {count_params(unet_model):,}")

## 2. Load Checkpoints & Test Data

In [ ]:
basic_ckpt = '../checkpoints/best_basic_cnn.pth'
unet_ckpt = '../checkpoints/best_unet.pth'

if os.path.exists(basic_ckpt):
    load_checkpoint(basic_ckpt, basic_model, device=device)
    print("Loaded Basic CNN checkpoint.")
else:
    print("Basic CNN checkpoint not found; model is untrained.")

if os.path.exists(unet_ckpt):
    load_checkpoint(unet_ckpt, unet_model, device=device)
    print("Loaded U-Net checkpoint.")
else:
    print("U-Net checkpoint not found; model is untrained.")

_, _, test_loader = get_dataloaders('../data/processed', batch_size=8)

## 3. Visual & Numerical Evaluation

In [ ]:
basic_model.eval()
unet_model.eval()

batch = next(iter(test_loader))
l_t = batch['L'].to(device)
rgb_t = batch['rgb']

with torch.no_grad():
    pred_basic_ab = basic_model(l_t)
    pred_unet_ab = unet_model(l_t)

pred_basic_rgb = lab_to_rgb(l_t, pred_basic_ab)
pred_unet_rgb = lab_to_rgb(l_t, pred_unet_ab)
gt_rgb = (rgb_t.permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)

# Display first test sample
idx = 0
sample_gt = gt_rgb[idx]
sample_l = (l_t[idx, 0].cpu().numpy() * 255.0).astype(np.uint8)
sample_basic = pred_basic_rgb[idx]
sample_unet = pred_unet_rgb[idx]

m_basic = calculate_all_metrics(sample_basic, sample_gt)
m_unet = calculate_all_metrics(sample_unet, sample_gt)

print("Sample Metrics:")
print(f"Basic CNN -> MAE: {m_basic['MAE']:.2f}, PSNR: {m_basic['PSNR']:.2f} dB, SSIM: {m_basic['SSIM']:.4f}")
print(f"U-Net     -> MAE: {m_unet['MAE']:.2f}, PSNR: {m_unet['PSNR']:.2f} dB, SSIM: {m_unet['SSIM']:.4f}")

plot_comparison_grid(sample_gt, sample_l, sample_basic, sample_unet, m_basic, m_unet)